##1. LDA 기반 토픽 모델링 (20 뉴스그룹)

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 모터사이클, 야구, 그래픽스, 윈도우즈, 중동, 기독교, 전자공학, 의학 8개 주제를 추출.
cats = ['rec.motorcycles', 'rec.sport.baseball', 'comp.graphics', 'comp.windows.x',
        'talk.politics.mideast', 'soc.religion.christian', 'sci.electronics', 'sci.med']

# 위에서 cats 변수로 기재된 카테고리만 추출. subset='all'로 오타 수정
news_df = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'),
                             categories=cats, random_state=0)

# LDA는 Count 기반의 벡터화만 적용. max_features=1000 오타 수정
count_vect = CountVectorizer(max_df=0.95, max_features=1000, min_df=2, stop_words='english', ngram_range=(1, 2))
feat_vect = count_vect.fit_transform(news_df.data)
print('CountVectorizer Shape:', feat_vect.shape)

# LDA 모델 객체 생성 및 학습
lda = LatentDirichletAllocation(n_components=8, random_state=0)
lda.fit(feat_vect)

print(lda.components_.shape)

# 토픽별 핵심 단어 출력 함수
def display_topics(model, feature_names, no_top_words):
    for topic_index, topic in enumerate(model.components_):
        print('Topic #', topic_index)
        # components_ array에서 가장 값이 큰 순으로 정렬했을 때, 그 값의 array 인덱스를 반환.
        topic_word_indexes = topic.argsort()[::-1]
        top_indexes = topic_word_indexes[:no_top_words]

        # top_indexes 대상인 인덱스별로 feature_names에 해당하는 word feature 추출 후 join으로 concat
        feature_concat = ' '.join([feature_names[i] for i in top_indexes])
        print(feature_concat)

# CountVectorizer 객체 내의 전체 word의 명칭을 get_feature_names_out()를 통해 추출 (최신 버전 반영)
feature_names = count_vect.get_feature_names_out()

# 토픽별 가장 연관도가 높은 word를 15개만 추출
display_topics(lda, feature_names, 15)

CountVectorizer Shape: (7862, 1000)
(8, 1000)
Topic # 0
10 year medical health 1993 20 12 disease cancer team patients research number new 11
Topic # 1
don just like know think good time ve does way really people want ll right
Topic # 2
image file jpeg output program gif images format files color entry use bit 03 02
Topic # 3
armenian armenians turkish people said turkey armenia government genocide turks muslim russian greek azerbaijan killed
Topic # 4
israel jews dos jewish israeli dos dos arab state people arabs palestinian adl ed anti peace
Topic # 5
edu com available graphics ftp window use mail data motif software version pub information server
Topic # 6
god people jesus church believe say christ does christian think christians did know bible man
Topic # 7
thanks use using does help like display need problem know server screen windows window program


##2. Opinion Review 데이터 세트 로딩 및 문서 군집화 (K-Means)

In [2]:
# 1. 데이터 세트 zip 파일 다운로드
!wget https://archive.ics.uci.edu/static/public/288/opinosis+opinion+frasl+review.zip

# 2. 압축 해제 (opinosis_data 폴더 내에 저장)
!unzip -q opinosis+opinion+frasl+review.zip -d opinosis_data

--2026-05-26 12:19:29--  https://archive.ics.uci.edu/static/public/288/opinosis+opinion+frasl+review.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-26 12:19:30 ERROR 404: Not Found.

unzip:  cannot find or open opinosis+opinion+frasl+review.zip, opinosis+opinion+frasl+review.zip.zip or opinosis+opinion+frasl+review.zip.ZIP.


In [3]:
import pandas as pd
import glob, os
import warnings
import nltk
import string
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 700)

# 다운로드에 필요한 nltk 데이터
nltk.download('punkt')
nltk.download('wordnet')

# 책 부록에 생략된 LemNormalize 함수 구현 (실행을 위해 추가)
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
lemmar = WordNetLemmatizer()

def LemTokens(tokens):
    return [lemmar.lemmatize(token) for token in tokens]

def LemNormalize(text):
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))

path = '/content/opinosis_data/OpinosisDataset1.0/topics'

all_files = glob.glob(os.path.join(path, "*.data"))
filename_list = []
opinion_text = []

# 파일 취합 및 DataFrame 로딩
for file_ in all_files:
    df = pd.read_table(file_, index_col=None, header=0, encoding='latin1')
    # 코랩(Linux) 환경과 윈도우 환경 모두 동작하도록 os.path.basename 사용
    filename_ = os.path.basename(file_)
    filename = filename_.split('.')[0]

    filename_list.append(filename)
    opinion_text.append(df.to_string())

document_df = pd.DataFrame({'filename':filename_list, 'opinion_text':opinion_text})

# TF-IDF 피처 벡터화 적용
tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english',
                             ngram_range=(1,2), min_df=0.05, max_df=0.85)

# (주의: 실제 데이터가 있어야 아래 코드 실행 가능)
# feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

# 3개의 집합으로 군집화 수행
# km_cluster = KMeans(n_clusters=3, max_iter=10000, random_state=0)
# km_cluster.fit(feature_vect)
# document_df['cluster_label'] = km_cluster.labels_
# document_df.sort_values(by='cluster_label')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


##3. 군집별 핵심 단어 추출

In [4]:
# 군집별 top n 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명을 반환함.
def get_cluster_details(cluster_model, cluster_data, feature_names, clusters_num, top_n_features=10):
    cluster_details = {}
    # cluster_centers_ array의 값이 큰 순으로 정렬된 인덱스 값을 반환
    centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:, ::-1]

    for cluster_num in range(clusters_num):
        cluster_details[cluster_num] = {}
        cluster_details[cluster_num]['cluster'] = cluster_num

        top_feature_indexes = centroid_feature_ordered_ind[cluster_num, :top_n_features]
        top_features = [feature_names[ind] for ind in top_feature_indexes]
        top_feature_values = cluster_model.cluster_centers_[cluster_num, top_feature_indexes].tolist()

        # DataFrame에서 클러스터 번호에 해당하는 파일명 추출 (조건식 == 오타 수정)
        filenames = cluster_data[cluster_data['cluster_label'] == cluster_num]['filename']
        filenames = filenames.values.tolist()

        cluster_details[cluster_num]['top_features'] = top_features
        cluster_details[cluster_num]['top_features_value'] = top_feature_values
        cluster_details[cluster_num]['filenames'] = filenames

    return cluster_details

def print_cluster_details(cluster_details):
    for cluster_num, cluster_detail in cluster_details.items():
        print('####### Cluster {0}'.format(cluster_num))
        print('Top features:', cluster_detail['top_features'])
        print('Reviews 파일명:', cluster_detail['filenames'][:7])
        print('==================================================')

# (실제 실행을 위한 코드 예시 - 주석 처리됨)
# feature_names = tfidf_vect.get_feature_names_out()
# cluster_details = get_cluster_details(cluster_model=km_cluster, cluster_data=document_df,
#                                       feature_names=feature_names, clusters_num=3, top_n_features=10)
# print_cluster_details(cluster_details)

##4. 문서 유사도 측정 (Cosine Similarity)

In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# 1. 수식으로 직접 구현하는 코사인 유사도 함수
def cos_similarity(v1, v2):
    dot_product = np.dot(v1, v2)
    l2_norm = (np.sqrt(sum(np.square(v1))) * np.sqrt(sum(np.square(v2))))
    similarity = dot_product / l2_norm
    return similarity

doc_list = ['if you take the blue pill, the story ends',
            'if you take the red pill, you stay in Wonderland',
            'if you take the red pill, I show you how deep the rabbit hole goes']

tfidf_vect_simple = TfidfVectorizer()
feature_vect_simple = tfidf_vect_simple.fit_transform(doc_list)
print('feature_vect_simple shape:', feature_vect_simple.shape)

# 밀집 행렬 변환 후 수동 함수로 계산 테스트
feature_vect_dense = feature_vect_simple.todense()
vect1 = np.array(feature_vect_dense[0]).reshape(-1,)
vect2 = np.array(feature_vect_dense[1]).reshape(-1,)
similarity_simple = cos_similarity(vect1, vect2)
print('문장 1, 문장 2 Cosine 유사도: {0:.3f}'.format(similarity_simple))

# 2. 사이킷런의 cosine_similarity API 활용
similarity_simple_pair = cosine_similarity(feature_vect_simple[0], feature_vect_simple)
print('사이킷런 API 첫번째 문서 유사도:\n', similarity_simple_pair)

similarity_all_pair = cosine_similarity(feature_vect_simple, feature_vect_simple)
print('사이킷런 API 전체 문서 유사도:\n', similarity_all_pair)

# (참고) 이전 리뷰 데이터의 특정 군집 간 유사도 비교 시각화 (코드 템플릿)
# sorted_index = similarity_pair.argsort()[:, ::-1]
# sorted_index = sorted_index[:, 1:] # 자기 자신 제외
# sns.barplot(x='similarity', y='filename', data=hotel_1_sim_df)
# plt.title(comparison_docname)

feature_vect_simple shape: (3, 18)
문장 1, 문장 2 Cosine 유사도: 0.402
사이킷런 API 첫번째 문서 유사도:
 [[1.         0.40207758 0.40425045]]
사이킷런 API 전체 문서 유사도:
 [[1.         0.40207758 0.40425045]
 [0.40207758 1.         0.45647296]
 [0.40425045 0.45647296 1.        ]]


##5. 한글 형태소 분석 및 네이버 영화 평점 감성 분석

In [7]:
# 학습 데이터(ratings_train.txt) 및 테스트 데이터(ratings_test.txt) 다운로드
!wget https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
!wget https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt

--2026-05-26 12:22:14--  https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14628807 (14M) [text/plain]
Saving to: ‘ratings_train.txt’

ratings_train.txt   100%[===================>]  13.95M  --.-KB/s    in 0.1s    

2026-05-26 12:22:14 (108 MB/s) - ‘ratings_train.txt’ saved [14628807/14628807]

--2026-05-26 12:22:14--  https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4893335 (4.7M) [application/octet-

In [11]:
# 패키지 설치
!pip install konlpy

import pandas as pd
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# 데이터 로딩
try:
    train_df = pd.read_csv('ratings_train.txt', sep='\t', encoding='utf-8')
    test_df = pd.read_csv('ratings_test.txt', sep='\t', encoding='utf-8')
except FileNotFoundError:
    print("데이터 파일이 없습니다.")

# 전처리
if 'train_df' in locals():
    train_df = train_df.fillna(' ')
    train_df['document'] = train_df['document'].apply(lambda x: re.sub(r"\d+", " ", x))
    train_df.drop('id', axis=1, inplace=True)

    test_df = test_df.fillna(' ')
    test_df['document'] = test_df['document'].apply(lambda x: re.sub(r"\d+", " ", x))
    test_df.drop('id', axis=1, inplace=True)

# 토크나이저 설정
okt = Okt()
def tw_tokenizer(text):
    return okt.morphs(text)

# TF-IDF 벡터화 및 모델 학습 수행
tfidf_vect = TfidfVectorizer(tokenizer=tw_tokenizer, ngram_range=(1, 2), min_df=3, max_df=0.9)
tfidf_vect.fit(train_df['document'])
tfidf_matrix_train = tfidf_vect.transform(train_df['document'])

lg_clf = LogisticRegression(random_state=0, solver='liblinear')
params = {'C': [1, 3.5, 4.5, 5.5, 10]}

grid_cv = GridSearchCV(lg_clf, param_grid=params, cv=3, scoring='accuracy', verbose=1)
grid_cv.fit(tfidf_matrix_train, train_df['label'])
print(grid_cv.best_params_, round(grid_cv.best_score_, 4))

# 테스트 데이터 예측
tfidf_matrix_test = tfidf_vect.transform(test_df['document'])
best_estimator = grid_cv.best_estimator_
preds = best_estimator.predict(tfidf_matrix_test)
print('최종 정확도:', accuracy_score(test_df['label'], preds))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
{'C': 3.5} 0.8593
최종 정확도: 0.86172
